In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import accuracy_score, classification_report
from sklearn.ensemble import GradientBoostingClassifier

In [ ]:

# 1. Завантаження даних 
# (Переконайся, що файли train.csv та test.csv знаходяться у тій самій папці, що й зошит)
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')


In [ ]:

# 2. Підготовка ознак та цільової змінної
# Відкидаємо колонки, які важко або недоцільно використовувати без додаткової обробки
X = train.drop(columns=['Survived', 'PassengerId', 'Name', 'Ticket', 'Cabin'])
y = train['Survived']

# Дані для формування фінального submission
X_submission = test.drop(columns=['PassengerId', 'Name', 'Ticket', 'Cabin'])

# Розбиваємо дані на тренувальну та валідаційну вибірки
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)


In [ ]:

# 3. Налаштування препроцесора
numeric_features = ['Age', 'SibSp', 'Parch', 'Fare']
categorical_features = ['Pclass', 'Sex', 'Embarked']

# Конвеєр для числових даних: заповнення пропусків медіаною + стандартизація
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Конвеєр для категоріальних даних: заповнення пропусків найчастішим значенням + One-Hot Encoding
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# Об'єднання трансформерів
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])


In [ ]:

# 4. Створення загального пайплайну з класифікатором
clf = GradientBoostingClassifier(random_state=42)

pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                           ('classifier', clf)])


In [ ]:

# 5. Пошук по сітці (GridSearchCV) для оптимізації гіперпараметрів
param_grid = {
    'classifier__n_estimators': [50, 100, 200],
    'classifier__learning_rate': [0.01, 0.05, 0.1],
    'classifier__max_depth': [3, 4, 5]
}

# Ініціалізація та запуск навчання
grid_clf = GridSearchCV(pipeline, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid_clf.fit(X_train, y_train)


In [ ]:

# 6. Виведення результатів тренування та оцінка на валідації
print('Grid best parameters:', grid_clf.best_params_)
print('Grid best score (accuracy):', grid_clf.best_score_)

val_preds = grid_clf.predict(X_val)
print("\nValidation Accuracy: {:.3f}".format(accuracy_score(y_val, val_preds)))
print("\nClassification Report:\n", classification_report(y_val, val_preds))


In [ ]:

# 7. Формування та збереження файлу для Kaggle
test_pred = grid_clf.predict(X_submission)

submission = pd.DataFrame({
    'PassengerId': test['PassengerId'],
    'Survived': test_pred.astype(int)
})

submission.to_csv("submission.csv", index=False)
print("\nSubmission saved to: submission.csv")
print("Submission shape:", submission.shape)